In [14]:
import numpy as np
import pandas as pd
import torch
from scipy.optimize import linprog, minimize, LinearConstraint

In [ ]:
def implied_D_F(chain):

    c = chain[chain.cp_flag == "C"].groupby("K").agg(cb=("best_bid", "first"), ca=("best_offer", "first"))
    p = chain[chain.cp_flag == "P"].groupby("K").agg(pb=("best_bid", "first"), pa=("best_offer", "first"))
    m = c.join(p, how="outer").reset_index()
    liq = m[(m.cb > 0) & (m.pb > 0) & (m.ca > 0) & (m.pa > 0)]
    if len(liq) < 3:
        return None
    
    K = liq.K.values
    L = (liq.cb - liq.pa).values
    U = (liq.ca - liq.pb).values

    A_ub, b_ub = [], []
    for i in range(len(K)):
        A_ub.append([-1.0, -K[i], 1.0])
        b_ub.append(-L[i])

        A_ub.append([ 1.0,  K[i], 1.0])
        b_ub.append( U[i])

    res = linprog([0.0, 0.0, -1.0], A_ub=A_ub, b_ub=b_ub,
                  bounds=[(None, None)] * 3, method="highs")
    if res.success:
        A, B, t = res.x
    else:
        cmid = 0.5 * (liq.cb + liq.ca)
        pmid = 0.5 * (liq.pb + liq.pa)
        B, A = np.polyfit(K, (cmid - pmid).values, 1)
        t = float("nan")
    D = -B
    F = A / D
    if not (0 < D < 1.2 and F > 0):
        return None
    return D, F, m, float(t)


def arb_free_analytic_center(K, B, A, S0):
    # analytic center
    # decreasing in strike; convex; left boundary slope >= -1;  right slope <= 0;
    # max(0, S0-K, bid) <= C <= min(S0, ask)
    # 1. feasible point
    # 2. maximize Sum log(slack)
    K = np.asarray(K, float)
    Ch = np.asarray(A, float)
    Cl = np.asarray(B, float)
    N = len(K)
    M = np.zeros((N - 1 + N - 2 + 2, N))
    b = np.zeros(N - 1 + N - 2 + 2)

    # decreasing
    for i in range(N - 1):
        M[i, i] = -1.0
        M[i, i + 1] = 1.0

    # convex
    for i in range(1, N - 1):
        M[N - 1 + i - 1, i - 1] = -1.0 / (K[i] - K[i - 1])
        M[N - 1 + i - 1, i]     =  1.0 / (K[i] - K[i - 1]) + 1.0 / (K[i + 1] - K[i])
        M[N - 1 + i - 1, i + 1] = -1.0 / (K[i + 1] - K[i])

    # left slope >= -1
    M[N - 1 + N - 2, 0] =  1.0 / (K[1] - K[0])
    M[N - 1 + N - 2, 1] = -1.0 / (K[1] - K[0])
    b[N - 1 + N - 2] = 1.0

    # right slope <= 0
    M[N - 1 + N - 2 + 1, N - 2] = -1.0 / (K[N - 1] - K[N - 2])
    M[N - 1 + N - 2 + 1, N - 1] = 1.0 / (K[N - 1] - K[N - 2])
    b[N - 1 + N - 2 + 1] = 0.0

    lb = np.maximum.reduce([np.zeros(N), S0 - K, Cl])
    ub = np.minimum(np.full(N, S0), Ch)

    M = np.vstack([M, np.eye(N), -np.eye(N)])
    b = np.concatenate([b, ub, -lb])

    res = linprog(np.zeros(N), A_ub=M, b_ub=b, method="highs")
    # Initial guess for the center (e.g., origin)
    x0 = res.x if res.success else np.clip(0.5 * (Cl + Ch), lb, ub)

    obj = lambda x: -np.sum(np.log(np.clip(b - M @ x, 1e-12, None)))
    r = minimize(obj, x0, method="trust-constr", constraints=LinearConstraint(M, -np.inf, b),
                 options={"maxiter": 2000, "gtol": 1e-8, "xtol": 1e-10})
    return r.x

In [ ]:
def process_all(csv_path, date, quotes_out=None,
                arbfree_out=None, min_strikes=8, verbose=True):
    # process every expiry of one trade date
    if quotes_out  is None: 
        quotes_out  = f"Quotes_SPX_{date}.csv"
    if arbfree_out is None: 
        arbfree_out = f"ArbFree_SPX_{date}.csv"

    df = pd.read_csv(csv_path)
    df = df[df["date"] == date].copy()
    df["K"] = df["strike_price"] / 1000.0
    perexp = {}
    for exdate, chain in df.groupby("exdate"):
        r = implied_D_F(chain)
        if r is None:
            continue
        D, F, m, t = r
        T = np.busday_count(np.datetime64(pd.to_datetime(str(date),   format="%Y%m%d").date()),
                            np.datetime64(pd.to_datetime(str(exdate), format="%Y%m%d").date())) / 252.0
        if T > 0:
            perexp[int(exdate)] = (D, F, T, m, t)
    if not perexp:
        print("No usable expiries on", date)
        return None
    Ts  = np.array([v[2] for v in perexp.values()])
    DFs = np.array([v[0] * v[1] for v in perexp.values()])

    if len(Ts) >= 2:
        slope, intercept = np.polyfit(Ts, np.log(DFs), 1)
        S0, q = float(np.exp(intercept)), float(-slope)
    else:
        S0, q = float(DFs[0]), 0.0

    qrows, arows = [], []
    for exdate, (D, F, T, m, t) in sorted(perexp.items()):
        mfac = S0 / (D * F)
        Khat = m.K.values * S0 / F
        getq = lambda col, miss: np.where(np.nan_to_num(col, nan=0.0) > 0, np.nan_to_num(col, nan=0.0), miss)

        # missing ask -> F
        ca = getq(m.ca.values, F)
        pa = getq(m.pa.values, F)

        # missing bid -> 0
        cb = getq(m.cb.values, 0.0)
        pb = getq(m.pb.values, 0.0)

        intr = np.maximum(S0 - Khat, 0.0)
        A = np.minimum(mfac * ca, mfac * pa + (S0 - Khat))
        B = np.maximum.reduce([mfac * cb, mfac * pb + (S0 - Khat), intr])
        ok = (A >= B)
        Khat, A, B = Khat[ok], A[ok], B[ok]

        # sort by strike
        o = np.argsort(Khat)
        Khat, A, B = Khat[o], A[o], B[o]
        keep = np.concatenate([[True], np.diff(Khat) > 1e-9])
        Khat, A, B = Khat[keep], A[keep], B[keep]
        if len(Khat) < min_strikes:
            continue
        arb = arb_free_analytic_center(Khat, B, A, S0)
        for k, a, b, c in zip(Khat, A, B, arb):
            qrows.append((date, exdate, k, a, b, round(S0, 4), round(T, 6), round(D, 6), round(F, 4)))
            arows.append((date, exdate, c, k))
        if verbose:
            inb = float(((arb >= B - 1e-6) & (arb <= A + 1e-6)).mean())
            print(f"  exp {exdate} | T={T:.4f} D={D:.5f} F={F:.2f} | {len(Khat):3d} strikes | arb in-band {inb*100:.0f}%")

    pd.DataFrame(qrows, columns=["date", "exdate", "strike", "ask", "bid", "S0", "T", "D", "F"]).to_csv(quotes_out, index=False)
    pd.DataFrame(arows, columns=["date", "exdate", "price", "strike"]).to_csv(arbfree_out, index=False)
    if verbose:
        print(f"\\nSpot S0 = {S0:.3f} (implied q = {q:.4f}) -> wrote '{quotes_out}' and '{arbfree_out}'")
    return dict(S0=S0, q=q)

# moneyness=(0, np.inf) to keep all strikes
def extract(date, exdate, moneyness=(0, np.inf),
            quotes_file=None, arbfree_file=None):
    if quotes_file is None: 
        quotes_file  = f"Quotes_SPX_{date}.csv"
    if arbfree_file is None: 
        arbfree_file = f"ArbFree_SPX_{date}.csv"

    q = pd.read_csv(quotes_file)
    q = q[(q.date == date) & (q.exdate == exdate)].sort_values("strike")
    
    a = pd.read_csv(arbfree_file)
    a = a[(a.date == date) & (a.exdate == exdate)].sort_values("strike")
    if len(q) == 0:
        raise ValueError(f"no rows for date={date}, exdate={exdate} (run process_all first)")
    S0 = float(q.S0.iloc[0])
    q = q[(q.strike >= moneyness[0] * S0) & (q.strike <= moneyness[1] * S0)]
    a = a[(a.strike >= moneyness[0] * S0) & (a.strike <= moneyness[1] * S0)]
    K = q.strike.values
    bid_ask = torch.tensor(np.column_stack([K, q.bid.values, q.ask.values]), dtype=torch.float64)
    return dict(bid_ask=bid_ask,
                arb_p=torch.tensor(a.price.values, dtype=torch.float64),
                arb_K=torch.tensor(a.strike.values, dtype=torch.float64),
                S0=S0, T=float(q["T"].iloc[0]), D=float(q.D.iloc[0]), F=float(q.F.iloc[0]),
                R1=int((K <= S0).sum()), R2=int((K >= S0).sum()))

In [ ]:
process_all("SPX_opt_2011_2012.csv", 20110103, quotes_out="Quotes_SPX_20110103.csv", arbfree_out="ArbFree_SPX_20110103.csv")

ch = extract(20110103, 20110122)
bid_ask, arb_p, arb_K = ch["bid_ask"], ch["arb_p"], ch["arb_K"]
S0, T, D, F, R1, R2 = ch["S0"], ch["T"], ch["D"], ch["F"], ch["R1"], ch["R2"]
r = -np.log(D) / T
print(f"extracted: S0={S0:.3f}  T={T:.4f}  D={D:.5f}  r={r:+.4f}  R1={R1} R2={R2}  ({len(arb_K)} strikes)")

  exp 20110107 | T=0.0159 D=1.00167 F=1270.74 |  31 strikes | arb in-band 100%


/opt/anaconda3/lib/python3.13/site-packages/scipy/optimize/_differentiable_functions.py:376: UserWarning: delta_grad == 0.0. Check if the approximated function is linear. If the function is linear better results can be obtained by defining the Hessian as zero instead of using quasi-Newton approximations.
  self.H.update(self.x - self.x_prev, self.g - self.g_prev)


  exp 20110122 | T=0.0595 D=1.00045 F=1270.70 | 159 strikes | arb in-band 100%


/opt/anaconda3/lib/python3.13/site-packages/scipy/optimize/_differentiable_functions.py:376: UserWarning: delta_grad == 0.0. Check if the approximated function is linear. If the function is linear better results can be obtained by defining the Hessian as zero instead of using quasi-Newton approximations.
  self.H.update(self.x - self.x_prev, self.g - self.g_prev)


KeyboardInterrupt: 